In [1]:
import os
from scipy.io import loadmat
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
mat_files = [f for f in os.listdir('.') if f.endswith('_C_results.mat')]

data_dict = {}

for file in mat_files:
    data = loadmat(file)
    data_dict[file] = data  

for filename, content in data_dict.items():
    print(f"{filename}: keys = {list(content.keys())}")

94_C_results.mat: keys = ['__header__', '__version__', '__globals__', 'C_trace_region', 'C_trace_region2', 'timestamps', 'unique_area']
370_C_results.mat: keys = ['__header__', '__version__', '__globals__', 'C_trace_region', 'C_trace_region2', 'timestamps', 'unique_area']
369_C_results.mat: keys = ['__header__', '__version__', '__globals__', 'C_trace_region', 'C_trace_region2', 'timestamps', 'unique_area']
371_C_results.mat: keys = ['__header__', '__version__', '__globals__', 'C_trace_region', 'C_trace_region2', 'timestamps', 'unique_area']
367_C_results.mat: keys = ['__header__', '__version__', '__globals__', 'C_trace_region', 'C_trace_region2', 'timestamps', 'unique_area']


In [4]:
def get_data(file_name, time_index):
    mat = data_dict[file_name]
    trace = mat['C_trace_region2']  

    n_regions = 13
    region_trace = []
    
    for region_index in range(1, n_regions+1):
        cells_in_region = trace[time_index, :][region_index] 
        region_mean = np.mean(cells_in_region, axis=0)
        region_trace.append(region_mean)
    
    # region_trace shape = (n_regions, time)
    return np.vstack(region_trace)

        
def compute_correlation_matrix(data):
    # Compute the Pearson correlation matrix
    corr_matrix = np.corrcoef(data)
    return corr_matrix

In [5]:
mat = data_dict['94_C_results.mat']
trace = mat['C_trace_region2']
unique_area = mat['unique_area']
get_data('94_C_results.mat', 5).shape

(13, 601)

In [6]:
from scipy.signal import coherence

correlation_matrices = {}
all_trace = {}

File_list = ['94_C_results.mat', '370_C_results.mat', '369_C_results.mat', '371_C_results.mat', '367_C_results.mat']

from tqdm import tqdm

for file in tqdm(File_list, desc="Processing files"):
    for time_index in tqdm(range(1, 22), desc=f"{file}", leave=False):
        data = get_data(file, time_index)
        corr_matrix = compute_correlation_matrix(data)
        correlation_matrices[(file, time_index)] = corr_matrix
        all_trace[(file, time_index)] = data


94_C_results.mat:   0%|          | 0/21 [00:00<?, ?it/s]
                                                        
369_C_results.mat:   0%|          | 0/21 [00:00<?, ?it/s]
                                                         
Processing files: 100%|██████████| 5/5 [00:00<00:00, 16.93it/s]


In [7]:
correlation_matrix2 = correlation_matrices[('94_C_results.mat', 20)]
correlation_matrix2.shape

(13, 13)

In [8]:
cell_trace = all_trace[('94_C_results.mat', 1)]
cell_trace.shape

(13, 2401)

In [9]:
df = pd.read_excel('SEU_net_wo_corticalLayers.xlsx', header=1, index_col=1)
df = df.iloc[:, 1:]
df

,FRP,MOp,MOs,SSp-n,SSp-bfd,SSp-ll,SSp-m,SSp-ul,SSp-tr,SSp-un,...,MD,SMT,PVT,RE,CM,RT,PSV,PB,TRN,PRNr
FRP,0.000000,0.000000,2.028292,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0,0,0,0.000000
MOp,0.625221,1.992551,1.765525,0.930700,0.678581,1.968629,1.024268,0.953173,1.600363,0.961477,...,0.000000,0.000000,0.0,0.000000,0.0,0.861023,0,0,0,0.875238
MOs,1.417198,1.381516,1.645074,0.670320,2.725806,0.853611,1.209884,1.446972,0.000000,0.000000,...,0.000000,1.621683,0.0,0.000000,0.0,0.768877,0,0,0,0.000000
SSp-n,0.000000,0.789537,1.283165,1.795210,1.301726,0.000000,1.947067,2.399795,0.000000,1.158404,...,0.000000,0.000000,0.0,0.000000,0.0,0.810396,0,0,0,0.000000
SSp-bfd,0.000000,1.159698,0.766737,1.669994,1.934520,0.447926,1.166723,0.869816,3.467967,1.047941,...,0.000000,0.000000,0.0,0.000000,0.0,0.191446,0,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RT,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.514522,2.131420,0.0,4.582369,0.0,0.762490,0,0,0,0.000000
PSV,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0,0,0,0.720976
PB,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0,0,0,1.214098
TRN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0,0,0,0.000000


In [10]:
desired_order = ['MOp','MOs','SSp-bfd','SSp-ll','SSp-ul','SSp-tr','SSp-un',
                 'VISam','VISp','VISpm','RSPagl','RSPd','VISa']
label_list = [re.sub(r'1$', '', label[0]) for label in unique_area[0]]
filtered_df = df.loc[df.index.intersection(label_list), df.columns.intersection(label_list)]
filtered_df_ordered = filtered_df.loc[desired_order, desired_order]
filtered_df_ordered

,MOp,MOs,SSp-bfd,SSp-ll,SSp-ul,SSp-tr,SSp-un,VISam,VISp,VISpm,RSPagl,RSPd,VISa
MOp,1.992551,1.765525,0.678581,1.968629,0.953173,1.600363,0.961477,0.000000,0.335160,0.000000,2.211280,5.063436,0.000000
MOs,1.381516,1.645074,2.725806,0.853611,1.446972,0.000000,0.000000,0.000000,0.867938,0.000000,0.000000,0.000000,0.000000
SSp-bfd,1.159698,0.766737,1.934520,0.447926,0.869816,3.467967,1.047941,0.348838,0.480395,0.000000,0.461558,0.000000,1.000840
SSp-ll,1.950709,0.689180,0.643465,3.108903,1.395424,2.154667,0.864234,0.000000,0.000000,0.000000,0.712695,0.000000,0.000000
SSp-ul,1.443828,3.054838,0.598507,1.711227,2.481238,0.000000,1.348156,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
SSp-tr,1.591901,1.958032,1.266143,9.307183,11.478469,0.423247,1.219740,0.000000,1.153103,0.000000,6.914832,0.000000,0.000000
SSp-un,0.588739,0.540600,2.047131,0.393314,1.983733,0.309392,1.666413,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
VISam,0.000000,0.000000,0.000000,0.000000,0.000000,1.517546,0.000000,6.803200,0.441544,1.633869,2.684360,0.285605,0.000000
VISp,0.000000,0.000000,0.730200,0.000000,0.000000,0.000000,0.000000,4.966288,2.843499,0.000000,0.409365,0.000000,0.000000
VISpm,0.000000,0.000000,0.480395,0.000000,0.000000,0.000000,0.000000,0.715855,0.842735,2.217881,0.000000,0.000000,0.000000


In [11]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

scaler = MinMaxScaler()

scaled_array = scaler.fit_transform(filtered_df_ordered.values)
scaled_df = pd.DataFrame(scaled_array, index=filtered_df_ordered.index, columns=filtered_df_ordered.columns)
scaled_df

,MOp,MOs,SSp-bfd,SSp-ll,SSp-ul,SSp-tr,SSp-un,VISam,VISp,VISpm,RSPagl,RSPd,VISa
MOp,1.000000,0.577944,0.248947,0.211517,0.083040,0.461470,0.576974,0.000000,0.117869,0.00000,0.319788,1.000000,0.000000
MOs,0.693340,0.538514,1.000000,0.091715,0.126060,0.000000,0.000000,0.000000,0.305236,0.00000,0.000000,0.000000,0.000000
SSp-bfd,0.582017,0.250991,0.709706,0.048127,0.075778,1.000000,0.628860,0.051276,0.168945,0.00000,0.066749,0.000000,1.000000
SSp-ll,0.979001,0.225603,0.236064,0.334033,0.121569,0.621305,0.518619,0.000000,0.000000,0.00000,0.103068,0.000000,0.000000
SSp-ul,0.724613,1.000000,0.219571,0.183861,0.216165,0.000000,0.809016,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
SSp-tr,0.798926,0.640961,0.464502,1.000000,1.000000,0.122045,0.731955,0.000000,0.405523,0.00000,1.000000,0.000000,0.000000
SSp-un,0.295470,0.176965,0.751019,0.042259,0.172822,0.089214,1.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
VISam,0.000000,0.000000,0.000000,0.000000,0.000000,0.437589,0.000000,1.000000,0.155282,0.73668,0.388203,0.056405,0.000000
VISp,0.000000,0.000000,0.267884,0.000000,0.000000,0.000000,0.000000,0.729993,1.000000,0.00000,0.059201,0.000000,0.000000
VISpm,0.000000,0.000000,0.176240,0.000000,0.000000,0.000000,0.000000,0.105223,0.296373,1.00000,0.000000,0.000000,0.000000


In [15]:
import pandas as pd
import numpy as np
import pickle
import SimpleITK as sitk
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os
from iblatlas.regions import BrainRegions
import torch
import torch.nn as nn
import torch.optim as optim
import random
from tqdm import tqdm
import torch.nn.functional as F
from torch.optim import lr_scheduler
import time

# ----------------- Seed -----------------
def set_seed(seed=42):
    import random, os
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

# ----------------- Graph-leaning -----------------
class SimpleGraphLearn(nn.Module):
    def __init__(self, node_dim, nodes, dropout=0.5, struct_connectome=None):
        super(SimpleGraphLearn, self).__init__()
        self.nodes = nodes
        self.dropout = dropout
        
        self.gate_net = nn.Sequential(
            nn.Linear(node_dim * 2, node_dim),
            nn.Sigmoid()
        )
        
        self.attention = nn.Linear(node_dim, 1, bias=False)
        self.norm = nn.LayerNorm(nodes)
        
        self.w = nn.Parameter(torch.zeros(size=(nodes, node_dim)))
        nn.init.xavier_uniform_(self.w.data, gain=1.414)
        
        if struct_connectome is not None:
            self.struct_connectome = nn.Parameter(
                torch.FloatTensor(struct_connectome), 
                requires_grad=False
            )
        else:
            self.struct_connectome = None
            
        self.graph_weight = nn.Parameter(torch.ones(1))
        self.struct_weight = nn.Parameter(torch.ones(1))
        
    def forward(self, node_features, temporal_features=None, batch_size=1):
        batch_size, nodes, node_dim = node_features.shape
        
        if temporal_features is not None:
            concat_features = torch.cat([node_features, temporal_features], dim=-1)  # (B, N, 2*D)
            gate = self.gate_net(concat_features)  # (B, N, D)
            node_features = gate * node_features + (1 - gate) * temporal_features
        
        node_res = torch.einsum('bnd, nl -> bnl', node_features, self.w) + node_features
        skip_atten = torch.einsum('bnd,bdm->bnm', node_res, node_res.transpose(-1, -2))
        skip_atten = self.norm(skip_atten)
        
        attention_scores = torch.matmul(node_features, node_features.transpose(-2, -1))  # (B, N, N)
        attention_scores = attention_scores / (node_dim ** 0.5)
        attention_scores = F.dropout(attention_scores, self.dropout, training=self.training)
        
        adj_raw = attention_scores + skip_atten
        
        if self.struct_connectome is not None:
            struct_adj = self.struct_connectome.unsqueeze(0).expand(batch_size, -1, -1)
            adj_dynamic = self.graph_weight * adj_raw + self.struct_weight * struct_adj
        else:
            adj_dynamic = adj_raw
            
        adj_dynamic = torch.tanh(adj_dynamic)
        
        if self.struct_connectome is not None:
            struct_upper_bound = self.struct_connectome.unsqueeze(0).expand(batch_size, -1, -1)
            adj_dynamic = torch.clamp(adj_dynamic, -struct_upper_bound, struct_upper_bound)
        
        return adj_dynamic

# ----------------- Temporal dynamic graph learning model -----------------
class TimeSeriesDynamicGraphModel(nn.Module):
    def __init__(self, device, num_nodes, seq_length=12, in_dim=1, out_dim=1, 
                 hidden_dim=32, dropout=0.3, embed_dim=10, heads=4, head_dim=8, 
                 struct_connectome=None):
        super(TimeSeriesDynamicGraphModel, self).__init__()
        self.device = device
        self.num_nodes = num_nodes
        self.seq_length = seq_length
        self.hidden_dim = hidden_dim
        self.dropout = dropout

        self.node_embedding = nn.Embedding(num_nodes, embed_dim)
        
        self.temporal_fusion = nn.Linear(hidden_dim, embed_dim)
        
        self.graph_learn = SimpleGraphLearn(
            node_dim=embed_dim, 
            nodes=num_nodes,
            dropout=dropout,
            struct_connectome=struct_connectome
        )

        self.temporal_encoder = nn.LSTM(
            input_size=in_dim, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True,
            dropout=dropout
        )

        self.gcn_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(2)
        ])

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, input, pred_time_embed=None):
        batch_size, seq_length, num_nodes, in_dim = input.shape
        
        idx = torch.arange(num_nodes).to(self.device)
        node_embeddings = self.node_embedding(idx)  # (num_nodes, embed_dim)
        node_embeddings = node_embeddings.unsqueeze(0).expand(batch_size, -1, -1)  # (batch_size, num_nodes, embed_dim)
        
        x_reshaped = input.permute(0, 2, 1, 3).contiguous().view(batch_size * num_nodes, seq_length, in_dim)
        temporal_out, _ = self.temporal_encoder(x_reshaped)
        
        temporal_features = temporal_out.view(batch_size, num_nodes, seq_length, -1)
        
        dynamic_graphs = []
        temporal_hiddens = []
        
        for t in range(seq_length):
            temporal_features_t = temporal_features[:, :, t, :]  # (batch_size, num_nodes, hidden_dim)
            
            temporal_features_proj_t = self.temporal_fusion(temporal_features_t)  # (batch_size, num_nodes, embed_dim)
            
            adj_dynamic_t = self.graph_learn(
                node_embeddings, 
                temporal_features_proj_t, 
                batch_size
            )  # (batch_size, num_nodes, num_nodes)
            
            dynamic_graphs.append(adj_dynamic_t)
            temporal_hiddens.append(temporal_features_t)
        
        adj_dynamic_all = torch.stack(dynamic_graphs, dim=1)  # (batch_size, seq_length, num_nodes, num_nodes)
        
        gcn_outputs_all = []
        
        for t in range(seq_length):
            h_t = temporal_hiddens[t]  # (batch_size, num_nodes, hidden_dim)
            adj_t = dynamic_graphs[t]  # (batch_size, num_nodes, num_nodes)
            
            gcn_outputs_t = []
            current_h = h_t
            for gcn_layer in self.gcn_layers:
                h_gcn = torch.matmul(adj_t.transpose(-1, -2), current_h)  
                h_gcn = gcn_layer(h_gcn)
                h_gcn = F.relu(h_gcn)
                h_gcn = F.dropout(h_gcn, self.dropout, training=self.training)
                gcn_outputs_t.append(h_gcn)
                current_h = h_gcn
            
            gcn_outputs_all.append(current_h)
        
        gcn_outputs_all = torch.stack(gcn_outputs_all, dim=1)  # (batch_size, seq_length, num_nodes, hidden_dim)
        
        final_hidden = gcn_outputs_all[:, -1, :, :]  # (batch_size, num_nodes, hidden_dim)
        output = self.output_layer(final_hidden)  # (batch_size, num_nodes, out_dim)
        
        return output, None, adj_dynamic_all

# ----------------- Trainer -----------------
class trainer():
    def __init__(self, num_nodes, seq_length, in_dim, out_dim, hidden_dim, dropout, lrate, wdecay, device, 
                 embed_dim, heads=4, head_dim=8, struct_connectome=None, struct_constraint_weight=0.2):
        
        self.model = TimeSeriesDynamicGraphModel(
            device, num_nodes, seq_length, in_dim, out_dim,
            hidden_dim=hidden_dim, dropout=dropout, embed_dim=embed_dim,
            heads=heads, head_dim=head_dim, struct_connectome=struct_connectome
        )
        
        self.model.to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lrate, weight_decay=wdecay)
        self.scheduler = lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=10, eps=0.00001, cooldown=20, verbose=True
        )
        
        self.loss = nn.MSELoss()
        self.clip = 0.5
        self.struct_constraint_weight = struct_constraint_weight

    def train(self, input, real_val, iter=0):
        self.model.train()
        output, gl_loss, adj_d = self.model(input)
        
        loss = self.loss(output, real_val)
        
        if hasattr(self.model.graph_learn, 'struct_connectome') and \
           self.model.graph_learn.struct_connectome is not None:
            
            # adj_d (batch_size, seq_length, num_nodes, num_nodes)
            if len(adj_d.shape) == 4:  # (batch_size, seq_length, num_nodes, num_nodes)
                avg_adj = torch.mean(adj_d, dim=(0, 1))  
            elif len(adj_d.shape) == 3:  # (batch_size, num_nodes, num_nodes)
                avg_adj = torch.mean(adj_d, dim=0)
            else:
                avg_adj = adj_d
            
            struct_adj = self.model.graph_learn.struct_connectome
            
            avg_adj_abs = torch.abs(avg_adj)
            struct_loss = F.mse_loss(avg_adj_abs, struct_adj)
            
            loss = loss + self.struct_constraint_weight * struct_loss
        
        loss.backward()
        
        if self.clip is not None:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clip)
        
        self.optimizer.step()
        self.optimizer.zero_grad()
        
        return loss.item()

    def eval(self, input, real_val):
        self.model.eval()
        with torch.no_grad():
            output, gl_loss, adj_d = self.model(input)
            loss = self.loss(output, real_val)
        return loss.item(), output, adj_d

# ----------------- Main training function -----------------
def train_timeseries_dynamic_graph(
    cell_trace, struct_connectome, seq_length, pred_length,
    hidden_dim, lr, epochs, batch_size, dropout,
    device=None, seed=42, val_ratio=0.2, patience=20,
    heads=4, head_dim=8, embed_dim=10, struct_constraint_weight=0.5,
    save_dir="./results/wo_layer/stage1"   
):

    import os, numpy as np, torch, random
    from tqdm import tqdm

    set_seed(seed)
    # Create save folder
    os.makedirs(save_dir, exist_ok=True)

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    num_regions, total_time = cell_trace.shape
    if seq_length is None:
        seq_length = total_time - pred_length

    # ------------------ Data input ------------------
    data, targets = [], []
    cell_trace = (cell_trace - np.mean(cell_trace)) / (np.std(cell_trace) + 1e-8)

    for t in range(total_time - seq_length - pred_length + 1):
        x = cell_trace[:, t:t+seq_length].T.reshape(seq_length, num_regions, 1)
        y = cell_trace[:, t+seq_length:t+seq_length+pred_length].T.reshape(num_regions, pred_length)
        data.append(x); targets.append(y)

    data, targets = np.array(data), np.array(targets)
    split = int(len(data) * (1 - val_ratio))

    train_data = torch.FloatTensor(data[:split]).to(device)
    train_targets = torch.FloatTensor(targets[:split]).to(device)
    val_data = torch.FloatTensor(data[split:]).to(device)
    val_targets = torch.FloatTensor(targets[split:]).to(device)

    # ------------------ Model ------------------
    engine = trainer(
        num_nodes=num_regions, seq_length=seq_length, in_dim=1, out_dim=pred_length,
        hidden_dim=hidden_dim, dropout=dropout, lrate=lr, wdecay=0.0, device=device,
        embed_dim=embed_dim, heads=heads, head_dim=head_dim,
        struct_connectome=struct_connectome, struct_constraint_weight=struct_constraint_weight
    )

    best_val = float('inf'); patience_count = 0
    train_losses, val_losses = [], []
    epoch_avg_graphs = []

    print(f"Training start | seq={seq_length}, T={total_time}")

    # ------------------ Train ------------------
    for epoch in range(epochs):
        engine.model.train()
        train_loss = 0
        for i in range(0, len(train_data), batch_size):
            train_loss += engine.train(train_data[i:i+batch_size], train_targets[i:i+batch_size])
        train_loss /= (len(train_data)//batch_size + 1)
        train_losses.append(train_loss)

        # -------- Validation --------
        engine.model.eval()
        val_loss = 0
        with torch.no_grad():
            for i in range(0, len(val_data), batch_size):
                val_loss += engine.eval(val_data[i:i+batch_size], val_targets[i:i+batch_size])[0]
        val_loss /= (len(val_data)//batch_size + 1)
        val_losses.append(val_loss)
        engine.scheduler.step(val_loss)

        # ---- Save best ----
        if val_loss < best_val:
            best_val = val_loss
            patience_count = 0
            torch.save(engine.model.state_dict(), f"{save_dir}/best_model.pth")  
        else:
            patience_count += 1
        if patience_count >= patience:
            print(f"Early stop @ epoch {epoch+1}")
            break

        # ---- collect epoch avg graph ----
        with torch.no_grad():
            x = train_data[0:1]
            _, _, adj = engine.model(x)
            epoch_avg_graphs.append(np.mean(adj.squeeze(0).cpu().numpy(), axis=0))

        if (epoch+1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train {train_loss:.5f} | Val {val_loss:.5f}")

    print("Training done, loading best model")
    engine.model.load_state_dict(torch.load(f"{save_dir}/best_model.pth"))
    engine.model.eval()

    np.save(f"{save_dir}/train_losses.npy", np.array(train_losses))
    np.save(f"{save_dir}/val_losses.npy", np.array(val_losses))
    np.save(f"{save_dir}/epoch_avg_graphs.npy", np.array(epoch_avg_graphs))
    print(f"Saved epoch_avg_graphs.npy to {save_dir}")

    # ------------------ Full time inference ------------------
    print("Reconstructing full dynamic graph...")

    full_dynamic_graphs = np.zeros((total_time, num_regions, num_regions))
    frame_count = np.zeros(total_time)

    with torch.no_grad():
        for t in tqdm(range(total_time - seq_length + 1)):
            x = cell_trace[:, t:t+seq_length].T.reshape(1, seq_length, num_regions, 1)
            x = torch.FloatTensor(x).to(device)

            _, _, adj_seq = engine.model(x)
            adj_seq = adj_seq.squeeze(0).cpu().numpy()

            for k in range(seq_length):
                idx = t + k
                full_dynamic_graphs[idx] += adj_seq[k]
                frame_count[idx] += 1

    for i in range(total_time):
        if frame_count[i] > 0:
            full_dynamic_graphs[i] /= frame_count[i]

    np.save(f"{save_dir}/dynamic_full_graphs.npy", full_dynamic_graphs)
    print(f"Saved dynamic_full_graphs.npy to {save_dir}, shape={full_dynamic_graphs.shape}")

    return engine.model, train_losses, val_losses, full_dynamic_graphs

In [28]:
structural_connectome_data = scaled_df.values

for i in [1,2,3,4,0]:
    
    mouse_index = i

    all_data = []
    for (file, time_index), data in all_trace.items():
        all_data.append(data)
    
    all_data_concatenated = []
    
    num_mice = len(all_data) // 21  
    
    for mice_num in range(num_mice):
        mice_data = all_data[mice_num * 21 : (mice_num + 1) * 21]
        mice_concatenated = np.concatenate(mice_data, axis=1)  # (num_regions, T * 21)
        all_data_concatenated.append(mice_concatenated)
    
    all_mice_concatenated = np.stack(all_data_concatenated, axis=0)  # (num_mice, num_regions, T * 21)
    
    selected_mouse_data = all_mice_concatenated[mouse_index][:, :]  
    min_vals = np.min(selected_mouse_data, axis=1, keepdims=True)  
    max_vals = np.max(selected_mouse_data, axis=1, keepdims=True)  
    
    selected_mouse_scaled_data = (selected_mouse_data - min_vals) / (max_vals - min_vals + 1e-6) 
    
    frame_counts = [2401]
    frame_counts += [201, 181, 21, 601] * 5
    
    stages_data = []
    current_frame = 0
    for stage_index, frame_count in enumerate(frame_counts):
        stage_data = selected_mouse_data[:, current_frame:current_frame + frame_count]
        stages_data.append(stage_data)
        
        current_frame += frame_count
    
    #for j, stage_data in enumerate(stages_data):
        #print(f"Stage {j} data shape: {stage_data.shape}")

    # ====== Date input ======
    for j in range(21):
        cell_trace = stages_data[j]            
        struct_connectome = structural_connectome_data
        if j == 0:
            seq_length = 100
            pred_length = 20
        else:
            seq_length = 10
            pred_length = 2
    
        # ====== Call the training function ======
        model, train_losses, val_losses, full_dynamic_graphs = train_timeseries_dynamic_graph(
            cell_trace = cell_trace,
            struct_connectome = structural_connectome_data,
            seq_length = seq_length,          
            pred_length = pred_length,         
            hidden_dim = 32,
            lr = 1e-3,
            epochs = 300,
            batch_size = 8,
            dropout = 0.6,
            device = None,
            seed = 42,
            val_ratio = 0.2,
            patience = 20,
            heads = 4,
            head_dim = 8,
            embed_dim = 10,
            struct_constraint_weight = 0.5,
            save_dir=f"./result/mouse_{mouse_index}/stage_{j}/"
        )

Training start | seq=100, T=2401
Epoch 20/300 | Train 0.82363 | Val 0.86596
Epoch 40/300 | Train 0.80484 | Val 0.84779
Epoch 60/300 | Train 0.79833 | Val 0.84712
Epoch 80/300 | Train 0.79546 | Val 0.84502
Epoch 100/300 | Train 0.78848 | Val 0.83138
Epoch 120/300 | Train 0.77922 | Val 0.82057
Early stop @ epoch 126
Training done, loading best model
Saved epoch_avg_graphs.npy to ./result/mouse_1/stage_0/
Reconstructing full dynamic graph...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2302/2302 [02:40<00:00, 14.31it/s]


Saved dynamic_full_graphs.npy to ./result/mouse_1/stage_0/, shape=(2401, 13, 13)
Training start | seq=10, T=201
Epoch 20/300 | Train 0.82123 | Val 1.09151
Epoch 40/300 | Train 0.77382 | Val 0.98236
Epoch 60/300 | Train 0.67037 | Val 0.91150
Epoch 80/300 | Train 0.64125 | Val 0.88322
Epoch 100/300 | Train 0.64085 | Val 0.85397
Epoch 120/300 | Train 0.63659 | Val 0.82911
Epoch 140/300 | Train 0.62487 | Val 0.82542
Epoch 160/300 | Train 0.62296 | Val 0.81147
Epoch 180/300 | Train 0.59467 | Val 0.80743
Epoch 200/300 | Train 0.59984 | Val 0.80479
Early stop @ epoch 219
Training done, loading best model
Saved epoch_avg_graphs.npy to ./result/mouse_1/stage_1/
Reconstructing full dynamic graph...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 192/192 [00:05<00:00, 37.48it/s]


Saved dynamic_full_graphs.npy to ./result/mouse_1/stage_1/, shape=(201, 13, 13)
Training start | seq=10, T=181
Epoch 20/300 | Train 0.89887 | Val 1.00953
Epoch 40/300 | Train 0.83254 | Val 0.94261
Epoch 60/300 | Train 0.78772 | Val 0.89563
Epoch 80/300 | Train 0.73800 | Val 0.86608
Epoch 100/300 | Train 0.72077 | Val 0.85727
Epoch 120/300 | Train 0.69064 | Val 0.86138
Epoch 140/300 | Train 0.70566 | Val 0.83456
Epoch 160/300 | Train 0.70396 | Val 0.82952
Epoch 180/300 | Train 0.69108 | Val 0.82490
Epoch 200/300 | Train 0.68357 | Val 0.82182
Epoch 220/300 | Train 0.67664 | Val 0.82216
Early stop @ epoch 223
Training done, loading best model
Saved epoch_avg_graphs.npy to ./result/mouse_1/stage_2/
Reconstructing full dynamic graph...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [00:04<00:00, 36.84it/s]


... (167 outputs left out)

Saved dynamic_full_graphs.npy to ./result/mouse_4/stage_19/, shape=(21, 13, 13)
Training start | seq=10, T=601


Epoch 20/300 | Train 0.55642 | Val 1.25663
Epoch 40/300 | Train 0.43516 | Val 0.95554
Epoch 60/300 | Train 0.38001 | Val 0.84629
Epoch 80/300 | Train 0.36736 | Val 0.76486
Epoch 100/300 | Train 0.34797 | Val 0.74562
Epoch 120/300 | Train 0.34219 | Val 0.71231
Epoch 140/300 | Train 0.34027 | Val 0.70906
Epoch 160/300 | Train 0.33420 | Val 0.69378
Epoch 180/300 | Train 0.33233 | Val 0.66710
Epoch 200/300 | Train 0.32493 | Val 0.65986
Early stop @ epoch 220
Training done, loading best model
Saved epoch_avg_graphs.npy to ./result/mouse_4/stage_20/
Reconstructing full dynamic graph...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 592/592 [00:05<00:00, 118.01it/s]

Saved dynamic_full_graphs.npy to ./result/mouse_4/stage_20/, shape=(601, 13, 13)
